# SQL reference 9: arrays, structs, maps and JSON

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-09-arrays-structs-and-json.ipynb)

Arrays (indexing, set operations, `explode` variants, higher-order functions), structs and the struct arg-max trick, maps, JSON extraction with paths and `from_json`/`to_json`, and Spark 4.0 `VARIANT`. Practise it in [SQL practice 9: arrays, structs, maps and JSON](sql-practice-09-arrays-structs-and-json.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose. This notebook also defines `posts` (tags as arrays) and `raw_events` (JSON payloads).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

# Semi-structured sample data for this notebook.
spark.sql("""
CREATE OR REPLACE TEMP VIEW posts AS SELECT * FROM VALUES
  (1, 'Spring planting guide', array('garden', 'howto', 'seasonal')),
  (2, 'Knife care',            array('kitchen', 'howto')),
  (3, 'New arrivals',          array()),
  (4, 'BBQ season',            array('outdoor', 'seasonal', 'garden'))
AS posts(post_id, title, tags)
""")
_ = spark.sql("""
CREATE OR REPLACE TEMP VIEW raw_events AS SELECT * FROM VALUES
  (1, '{"type": "checkout", "user": {"id": 7, "tier": "gold"}, "items": [{"sku": "SKU-4001", "qty": 2, "price": 12.5}, {"sku": "SKU-4007", "qty": 1, "price": 95.0}]}'),
  (2, '{"type": "checkout", "user": {"id": 9}, "items": [{"sku": "SKU-4003", "qty": 1, "price": 149.0}]}'),
  (3, '{"type": "page_view", "user": {"id": 7, "tier": "gold"}, "page": "/garden"}'),
  (4, '{"type": "checkout", "user": {"id": 11, "tier": "silver"}, "items": []}')
AS raw_events(event_id, payload)
""")

## Contents

| section | entries |
| --- | --- |
| [Arrays](#arrays) | [Building and reading arrays](#building-and-reading-arrays), [Set and sort operations on arrays](#set-and-sort-operations-on-arrays), [`explode` and friends](#explode-and-friends), [Higher-order functions](#higher-order-functions) |
| [Structs and maps](#structs-and-maps) | [Structs](#structs), [Maps](#maps) |
| [JSON](#json) | [Extracting from JSON strings](#extracting-from-json-strings), [Parsing with `from_json` and writing with `to_json`](#parsing-with-from_json-and-writing-with-to_json), [`VARIANT` (Spark 4.0)](#variant-spark-40) |

## Arrays

### Building and reading arrays

| function | returns |
| --- | --- |
| `array(a, b, ...)`, `sequence(start, stop [, step])`, `split(s, regex)` | a new array |
| `arr[i]` | element at **0-based** index `i`; an error when out of range in ANSI mode |
| `get(arr, i)` | element at 0-based index `i`, NULL when out of range |
| `element_at(arr, i)` / `try_element_at(arr, i)` | element at **1-based** index (negative counts from the end); error / NULL when out of range |
| `size(arr)` / `cardinality` / `array_size` | number of elements (`size(NULL)` is -1 unless ANSI mode, then NULL) |
| `array_contains(arr, x)`, `array_position(arr, x)` | membership / 1-based position (0 when absent) |
| `slice(arr, start, len)` | a sub-array (1-based start) |
| `array_join(arr, sep [, null_replacement])` | a delimited string |

**Notes:** in Spark 4.0's default ANSI mode, use `get` and `try_element_at` whenever an array can be shorter than
the index. Spark's `[]` index is 0-based while `element_at` is 1-based; Postgres and DuckDB arrays are 1-based,
BigQuery uses `arr[OFFSET(0)]` / `arr[ORDINAL(1)]`.

In [ ]:
show("""
SELECT post_id, tags, size(tags) AS n, get(tags, 0) AS first_tag, try_element_at(tags, -1) AS last_tag,
       array_contains(tags, 'howto') AS is_howto, array_position(tags, 'seasonal') AS seasonal_at,
       array_join(tags, ' | ') AS joined
FROM posts ORDER BY post_id
""")

### Set and sort operations on arrays

| function | returns |
| --- | --- |
| `array_distinct(arr)` | duplicates removed |
| `array_union(a, b)`, `array_intersect(a, b)`, `array_except(a, b)` | set operations (distinct results) |
| `arrays_overlap(a, b)` | whether they share an element |
| `array_sort(arr)` / `sort_array(arr, asc)` | sorted (NULLs last / first) |
| `flatten(arr_of_arrs)`, `concat(a, b)` | concatenation |
| `array_max(arr)`, `array_min(arr)` | extremes |
| `arrays_zip(a, b)` | array of structs pairing elements |

In [ ]:
show("""
SELECT array_distinct(array(3, 1, 3, 2)) AS distinct_, array_union(array(1, 2), array(2, 3)) AS union_,
       array_intersect(array(1, 2, 3), array(2, 3, 4)) AS intersect_, array_except(array(1, 2, 3), array(2)) AS except_,
       array_sort(array(3, NULL, 1)) AS sorted, flatten(array(array(1), array(2, 3))) AS flat, arrays_zip(array('a', 'b'), array(1, 2)) AS zipped
""")

### `explode` and friends

| function | produces |
| --- | --- |
| `explode(arr)` | one row per element; rows with an empty or NULL array **disappear** |
| `explode_outer(arr)` | like `explode`, but keeps those rows with a NULL element |
| `posexplode(arr)` | `(pos, col)`: the 0-based position and the element |
| `inline(arr_of_structs)` | one row per struct, one column per field |
| `LATERAL VIEW [OUTER] explode(arr) t AS x` | Hive-style syntax for the same, usable in `FROM` with several explodes |

**Notes:** exploding multiplies rows, just like a join; aggregate back with `collect_list`/`collect_set` (or
`array_sort(collect_list(...))` for a deterministic order). Snowflake uses `LATERAL FLATTEN`, BigQuery and Postgres
`UNNEST`.

In [ ]:
show("SELECT post_id, explode(tags) AS tag FROM posts ORDER BY post_id, tag")
show("SELECT post_id, tag FROM posts LATERAL VIEW OUTER explode(tags) t AS tag WHERE post_id IN (2, 3) ORDER BY post_id, tag")
show("SELECT post_id, pos, tag FROM posts LATERAL VIEW posexplode(tags) t AS pos, tag WHERE post_id = 1 ORDER BY pos")
show("SELECT tag, count(*) AS posts FROM posts LATERAL VIEW explode(tags) t AS tag GROUP BY tag ORDER BY posts DESC, tag")

### Higher-order functions

Lambda functions (`x -> expr`, `(x, i) -> expr`) process arrays without exploding them:

| function | returns |
| --- | --- |
| `transform(arr, x -> f(x))` | each element mapped |
| `filter(arr, x -> cond)` | elements satisfying `cond` |
| `exists(arr, x -> cond)`, `forall(arr, x -> cond)` | any / all elements satisfy `cond` |
| `aggregate(arr, start, (acc, x) -> merge [, acc -> finish])` / `reduce` | fold to one value |
| `zip_with(a, b, (x, y) -> f)` | combine two arrays element-wise |
| `array_sort(arr, (l, r) -> comparator)` | custom sort (return negative, 0 or positive) |

**Notes:** higher-order functions avoid the shuffle of an explode-and-regroup, so they are cheaper for per-row array
logic.

In [ ]:
show("""
SELECT transform(array(1, 2, 3), x -> x * 10) AS times_10,
       filter(array(5, 12, 8, 30), x -> x > 10) AS over_10,
       exists(array(1, NULL, 3), x -> x IS NULL) AS has_null,
       forall(array(2, 4, 6), x -> x % 2 = 0) AS all_even,
       aggregate(array(12.5, 95.0, 34.5), 0D, (acc, x) -> acc + x) AS total,
       zip_with(array(2, 1), array(12.5, 95.0), (q, p) -> q * p) AS line_totals,
       transform(array('a', 'b'), (x, i) -> concat(i, ':', x)) AS indexed
""")

## Structs and maps

### Structs

A `STRUCT` groups named fields in one value, like a row inside a column.

| expression | returns |
| --- | --- |
| `struct(a, b)` / `named_struct('x', a, 'y', b)` | a struct (field names from the columns or given explicitly) |
| `s.field`, `s['field']` | a field |
| `s.*` | all fields as columns |
| `max(struct(amount, order_id))` | structs compare field by field, so this returns the whole row with the largest amount (an arg-max) |
| `to_json(s)` | a JSON object string |

In [ ]:
show("""
SELECT customer_id, max(struct(amount, order_id, status)) AS biggest,
       max(struct(amount, order_id, status)).order_id AS biggest_order
FROM orders GROUP BY customer_id ORDER BY customer_id
""")
show("SELECT s.*, to_json(s) AS json FROM (SELECT named_struct('id', 7, 'tier', 'gold') AS s)")

### Maps

A `MAP<K, V>` holds key-value pairs with unique keys, useful for sparse attributes and tags.

| function | returns |
| --- | --- |
| `map(k1, v1, ...)`, `map_from_arrays(keys, values)`, `map_from_entries(arr_of_structs)`, `str_to_map(s, pair_delim, kv_delim)` | a map |
| `m[key]` / `element_at(m, key)` | the value, NULL when the key is missing (`element_at` errors in ANSI mode; use `try_element_at`) |
| `map_keys(m)`, `map_values(m)`, `map_entries(m)` | arrays |
| `map_contains_key(m, k)` | membership |
| `map_filter(m, (k, v) -> cond)`, `transform_values(m, (k, v) -> f)`, `transform_keys` | lambda processing |
| `explode(m)` | rows of `(key, value)` |

In [ ]:
show("""
WITH p AS (SELECT 'SKU-4001' AS sku, map('color', 'steel', 'size', 'M') AS attrs)
SELECT sku, attrs['color'] AS color, attrs['weight'] AS weight, map_keys(attrs) AS keys,
       map_contains_key(attrs, 'size') AS has_size, transform_values(attrs, (k, v) -> upper(v)) AS upper_values
FROM p
""")
show("SELECT key, value FROM (SELECT str_to_map('utm_source=mail&utm_medium=email', '&', '=') AS m) LATERAL VIEW explode(m) t AS key, value ORDER BY key")

## JSON

### Extracting from JSON strings

| function | returns |
| --- | --- |
| `get_json_object(json, '$.path')` | one value as a string (`$.user.id`, `$.items[0].sku`), NULL when the path is missing |
| `json_tuple(json, 'k1', 'k2', ...)` | several top-level fields at once (in `SELECT` or `LATERAL VIEW`) |
| `json_array_length(json_array)`, `json_object_keys(json)` | array length / top-level keys |
| `schema_of_json('sample')` | a DDL schema inferred from a literal sample value (not a column) |

**Notes:** path extraction re-parses the string on every call; when several fields are needed, parse once with
`from_json` (below). Snowflake uses `payload:user.id`, BigQuery `JSON_VALUE(payload, '$.user.id')`, Postgres
`payload->'user'->>'id'`.

In [ ]:
show("""
SELECT event_id, get_json_object(payload, '$.type') AS type, get_json_object(payload, '$.user.id') AS user_id,
       get_json_object(payload, '$.user.tier') AS tier, get_json_object(payload, '$.items[0].sku') AS first_sku,
       json_array_length(get_json_object(payload, '$.items')) AS n_items
FROM raw_events ORDER BY event_id
""")
show("""SELECT schema_of_json('{"type": "checkout", "user": {"id": 7}, "items": [{"sku": "A", "qty": 2}]}') AS inferred""")   # needs a literal sample

### Parsing with `from_json` and writing with `to_json`

`from_json(json, schema)` parses a JSON string into typed structs, arrays and maps in one pass; `to_json(value)`
serialises a struct, array or map back to a string. The schema is a DDL string such as
`'type STRING, user STRUCT<id: INT, tier: STRING>, items ARRAY<STRUCT<sku: STRING, qty: INT, price: DOUBLE>>'`.

**Notes:** fields missing from the JSON become NULL; fields not in the schema are ignored; values that do not fit the
schema type become NULL in the default `PERMISSIVE` mode. Declaring the schema explicitly is faster and safer than
inferring it.

In [ ]:
show("""
WITH parsed AS (
    SELECT event_id,
           from_json(payload, 'type STRING, user STRUCT<id: INT, tier: STRING>, items ARRAY<STRUCT<sku: STRING, qty: INT, price: DOUBLE>>') AS e
    FROM raw_events
)
SELECT event_id, e.type, e.user.id AS user_id, coalesce(e.user.tier, 'none') AS tier, size(e.items) AS n_items,
       aggregate(e.items, 0D, (acc, i) -> acc + i.qty * i.price) AS basket_value
FROM parsed ORDER BY event_id
""")
show("""
SELECT customer_id,
       to_json(named_struct('customer_id', customer_id, 'orders', array_sort(collect_list(named_struct('id', order_id, 'status', status))))) AS doc
FROM orders WHERE customer_id <= 2 GROUP BY customer_id ORDER BY customer_id
""")

### `VARIANT` (Spark 4.0)

`VARIANT` stores semi-structured data in an efficient binary form without a fixed schema: parse once with
`parse_json`, then extract typed values with paths.

| function | returns |
| --- | --- |
| `parse_json(s)` / `try_parse_json(s)` | a `VARIANT` (error / NULL on invalid JSON) |
| `variant_get(v, '$.path', 'type')` | the value cast to `type` (error if it cannot be cast) |
| `try_variant_get(v, '$.path', 'type')` | NULL instead of an error |
| `schema_of_variant(v)` | the inferred schema of a value |
| `to_json(v)` | back to a JSON string |

**Notes:** Databricks and Snowflake also accept `v:path` shorthand; open-source Spark 4.0 uses `variant_get`.
`VARIANT` suits data whose fields vary per record (event payloads, API responses).

In [ ]:
show("""
SELECT event_id,
       variant_get(v, '$.type', 'string') AS type,
       variant_get(v, '$.user.id', 'int') AS user_id,
       try_variant_get(v, '$.page', 'string') AS page,
       schema_of_variant(v) AS schema
FROM (SELECT event_id, parse_json(payload) AS v FROM raw_events)
ORDER BY event_id
""")

**Next:** practise these entries in [SQL practice 9: arrays, structs, maps and JSON](sql-practice-09-arrays-structs-and-json.ipynb).